In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        (os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
import numpy as np
import mne
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Dropout, Bidirectional, LSTM, Dense
from sklearn.model_selection import train_test_split

# --------------------------
# 1. Load and preprocess EEG
# --------------------------
folder = "/kaggle/input/datasets/prashastham/bci-competition-iv-dataset-2a/Train"

all_epochs, all_labels = [], []

for file in os.listdir(folder):
    if file.endswith(".gdf"):
        raw = mne.io.read_raw_gdf(os.path.join(folder, file), preload=True)
        raw.filter(8., 30., fir_design='firwin')   # Bandpass: Mu + Beta

        # ICA for artifact removal
        ica = mne.preprocessing.ICA(n_components=15, random_state=42, max_iter=800)
        ica.fit(raw)
        raw = ica.apply(raw)

        # Extract events
        events, event_dict = mne.events_from_annotations(raw)
        valid_events = {k: v for k, v in event_dict.items() if k in ['769','770','771','772']}
        if len(valid_events) == 0:
            continue

        # Epoching
        epochs = mne.Epochs(raw, events, event_id=valid_events, tmin=0.5, tmax=2.5,
                            baseline=None, preload=True, event_repeated='drop', on_missing='ignore')

        if len(epochs) == 0:
            continue

        X = epochs.get_data()  # (trials, channels, time)
        y = epochs.events[:, -1]

        # Reshape: (trials, time, channels)
        X = np.transpose(X, (0,2,1))
        all_epochs.append(X)
        all_labels.append(y)

# Combine subjects
X = np.concatenate(all_epochs, axis=0)
y = np.concatenate(all_labels, axis=0)

# Normalize each trial
X = (X - np.mean(X, axis=1, keepdims=True)) / np.std(X, axis=1, keepdims=True)

# Encode labels
le = LabelEncoder()
y = le.fit_transform(y)
y = to_categorical(y)

print("X shape:", X.shape, "y shape:", y.shape)  # (trials, time_points, channels)

# --------------------------
# 2. CNN + BiLSTM Hybrid Model
# --------------------------
model = Sequential()

# CNN layers to extract spatial features across channels
model.add(Conv1D(64, kernel_size=3, activation='relu', input_shape=(X.shape[1], X.shape[2])))
model.add(MaxPooling1D(pool_size=2))
model.add(Dropout(0.3))

model.add(Conv1D(128, kernel_size=3, activation='relu'))
model.add(MaxPooling1D(pool_size=2))
model.add(Dropout(0.3))

# BiLSTM layer to capture temporal patterns
model.add(Bidirectional(LSTM(100)))
model.add(Dropout(0.5))

# Output layer
model.add(Dense(y.shape[1], activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

# --------------------------
# 3. Train-test split
# --------------------------
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=100, batch_size=32, verbose=2)

2026-04-06 18:02:17.103488: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775498537.296077      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775498537.353781      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775498537.792953      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775498537.793006      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775498537.793010      55 computation_placer.cc:177] computation placer alr

Extracting GDF parameters from /kaggle/input/datasets/prashastham/bci-competition-iv-dataset-2a/Train/A01T.gdf...
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
EEG-Fz, EEG, EEG, EEG, EEG, EEG, EEG, EEG-C3, EEG, EEG-Cz, EEG, EEG-C4, EEG, EEG, EEG, EEG, EEG, EEG, EEG, EEG-Pz, EEG, EEG, EOG-left, EOG-central, EOG-right
Creating raw.info structure...


/usr/lib/python3.12/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Reading 0 ... 672527  =      0.000 ...  2690.108 secs...
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 8 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 8.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 7.00 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 413 samples (1.652 s)

Fitting ICA to data using 25 channels (please be patient, this may take a while)
Selecting by number: 15 components
Fitting ICA took 6.7s.
Applying ICA to Raw instance
    Transforming to ICA space (15 components)
    Zeroing out 0 ICA components
    Projecting back using 25 PCA components
Used Annotations descriptions: [np.str_('1023'), np.str_('1072'), np.str_('276'), np.str_('2

/usr/lib/python3.12/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


...

 Trainable params: 213,974 (835.84 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100


I0000 00:00:1775498644.517783     123 cuda_dnn.cc:529] Loaded cuDNN version 91002


65/65 - 8s - 118ms/step - accuracy: 0.2436 - loss: 1.6212 - val_accuracy: 0.2736 - val_loss: 1.5581
Epoch 2/100
65/65 - 1s - 21ms/step - accuracy: 0.2571 - loss: 1.5809 - val_accuracy: 0.2813 - val_loss: 1.5824
Epoch 3/100
65/65 - 1s - 22ms/step - accuracy: 0.2489 - loss: 1.5722 - val_accuracy: 0.2852 - val_loss: 1.5589
Epoch 4/100
65/65 - 1s - 22ms/step - accuracy: 0.2904 - loss: 1.5482 - val_accuracy: 0.2697 - val_loss: 1.5440
Epoch 5/100
65/65 - 1s - 22ms/step - accuracy: 0.2943 - loss: 1.5422 - val_accuracy: 0.2890 - val_loss: 1.5693
Epoch 6/100
65/65 - 1s - 21ms/step - accuracy: 0.3164 - loss: 1.5247 - val_accuracy: 0.3237 - val_loss: 1.5212
Epoch 7/100
65/65 - 1s - 22ms/step - accuracy: 0.3468 - loss: 1.4989 - val_accuracy: 0.3025 - val_loss: 1.5167
Epoch 8/100
65/65 - 1s - 22ms/step - accuracy: 0.3247 - loss: 1.5083 - val_accuracy: 0.2871 - val_loss: 1.5249
Epoch 9/100
65/65 - 1s - 22ms/step - accuracy: 0.3507 - loss: 1.4727 - val_accuracy: 0.3295 - val_loss: 1.4809
Epoch 10/100